In [1]:
import pandas as pd
import numpy as np

# Load the clean data from airbnb_eda.ipynb
df = pd.read_csv("listings_clean.csv")
print(df.shape)

# Prepared dataset 1: hotel vs home feature (from the EDA finding)
df["is_hotel"] = df["property_type"].str.contains("hotel", case=False, na=False)
print(df["is_hotel"].value_counts())
print(df.groupby("is_hotel")["short_stay"].mean().mul(100).round(1))

(29918, 84)
is_hotel
False    27632
True      2286
Name: count, dtype: int64
is_hotel
False    13.0
True     95.6
Name: short_stay, dtype: float64


In [2]:
# Prepared dataset 2: which candidate features are too similar? (correlation > 0.8)
candidates = ["log_price", "log_price_per_guest", "accommodates", "bedrooms", "beds", "bathrooms_num",
              "log_minimum_nights", "availability_365", "log_number_of_reviews", "log_number_of_reviews_ltm",
              "log_reviews_per_month", "log_calculated_host_listings_count", "n_amenities", "occupancy_rate"]
corr = df[candidates].corr()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print(pairs[pairs.abs() > 0.8].sort_values(ascending=False).round(2))

log_number_of_reviews_ltm  occupancy_rate           0.93
                           log_reviews_per_month    0.82
dtype: float64


In [3]:
# Prepared dataset 3: final clustering features (priced listings only)
cluster_features = ["log_price", "log_price_per_guest", "accommodates", "bedrooms", "beds", "bathrooms_num",
                    "log_minimum_nights", "availability_365", "log_number_of_reviews",
                    "log_number_of_reviews_ltm", "log_calculated_host_listings_count", "n_amenities"]

priced = df[df["price"].notna()].copy()
X = (priced[cluster_features] - priced[cluster_features].mean()) / priced[cluster_features].std()   # z-score
X["is_hotel"] = priced["is_hotel"].astype(int)
X = pd.concat([X, pd.get_dummies(priced["room_type"], prefix="room", dtype=int)], axis=1)

clustering = pd.concat([priced[["id", "neighbourhood_group_cleansed", "license_status", "short_stay"]], X], axis=1)
clustering.to_csv("prepared_clustering.csv", index=False)
print(clustering.shape)
print(list(X.columns))

(19396, 21)
['log_price', 'log_price_per_guest', 'accommodates', 'bedrooms', 'beds', 'bathrooms_num', 'log_minimum_nights', 'availability_365', 'log_number_of_reviews', 'log_number_of_reviews_ltm', 'log_calculated_host_listings_count', 'n_amenities', 'is_hotel', 'room_Entire home/apt', 'room_Hotel room', 'room_Private room', 'room_Shared room']


In [4]:
# Prepared dataset 4: yes/no "items" for association rules (all 29,918 listings)
df["price_band"] = pd.qcut(df["price"], 3, labels=["low", "mid", "high"]).astype(str).replace("nan", "unknown")
print(df.groupby("price_band")["price"].agg(["min", "max"]).round(0))

items = pd.get_dummies(df[["neighbourhood_group_cleansed", "room_type", "license_status", "price_band"]],
                       prefix=["boro", "room", "license", "price"]).astype(bool)
items["short_stay"] = df["short_stay"]
items["stay_30plus"] = ~df["short_stay"]
items["hotel"] = df["is_hotel"]
items["professional_host"] = df["professional_host"]
amen_cols = [c for c in df.columns if c.startswith("amen_")]
items = pd.concat([items, df[amen_cols].astype(bool)], axis=1)

items.to_csv("prepared_transactions.csv", index=False)
print(items.shape)

              min      max
price_band                
high        262.0  10000.0
low          18.0    131.0
mid         131.0    262.0
unknown       NaN      NaN
(29918, 60)
